In [ ]:
# FEDTREES — CELL 1: setup, federated tree harness, partition + centralized-LightGBM reproduction checks, timing run.
#   CPU runtime is enough (High-RAM helps); ~15-20 min. New notebook for WP4. Reads the benchmark splits and harness
#   READ-ONLY; writes only to results/fedtrees/. Prints relative paths only.
#   Federated tree methods (8 classes, CIC IoT-DIAD 2024, same splits, client partitions and seeds as the benchmark grid):
#     ens : one-shot ensemble - every client trains its own LightGBM (300 trees per class, lr 0.05, 63 leaves);
#           the server averages the clients' class probabilities, weighted by client size
#     cyc : cyclic boosting - one shared booster; clients take turns adding one boosting iteration on their own data
#           (30 cycles x 10 clients = 300 iterations, the centralized budget); only the model travels, never data
#   A lossless histogram-aggregation GBDT is equivalent to centralized histogram GBDT on the pooled data, so the
#   centralized LightGBM of the benchmark serves as its reference (checked for exact reproduction below).
import os, glob, json, time, sys, importlib, gc
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip -q install lightgbm'); import lightgbm as lgb
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
RV2 = f'{DRIVE_PATH}/results/v2'; FT = f'{DRIVE_PATH}/results/fedtrees'
for sub in ('code', 'check'): os.makedirs(f'{FT}/{sub}', exist_ok=True)
open('/content/fl_core_t.py', 'w').write(open(f'{V2D}/fl_core.py').read()); sys.path.insert(0, '/content')

TREE_CORE = r'''
"""fedtree_core.py - federated gradient-boosted trees on the leak-free CIC IoT-DIAD 2024 splits (8 classes)."""
import time, numpy as np, lightgbm as lgb
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
NC = 8; LAB = np.arange(NC)
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=LAB))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=LAB)]
BASE = dict(objective='multiclass', num_class=NC, learning_rate=0.05, num_leaves=63, verbose=-1, num_threads=0)

def class_weights(y_all):
    cnt = np.bincount(y_all, minlength=NC).astype(float)            # aggregate class counts (shareable statistic)
    w = np.where(cnt > 0, len(y_all) / (NC * np.maximum(cnt, 1)), 0.0); return w   # 'balanced' weights

def score(S, P_val, P_test):
    pv, pt = P_val.argmax(1), P_test.argmax(1)
    return {'val_macro_f1': mf1(S['y_val'], pv), 'test_macro_f1': mf1(S['y_test'], pt), 'test_accuracy': float(accuracy_score(S['y_test'], pt)),
            'test_per_class_f1': pcf(S['y_test'], pt), 'n_pred_classes': int(len(np.unique(pt)))}

def run_ens(S, parts, seed, n_trees=300):
    t0 = time.time(); w = class_weights(S['y_train']); n_k = np.array([len(p) for p in parts], float); a = n_k / n_k.sum()
    Pv = np.zeros((len(S['y_val']), NC)); Pt = np.zeros((len(S['y_test']), NC)); classes_seen = []
    for k, p in enumerate(parts):
        Xk, yk = S['X_train'][p], S['y_train'][p]
        bst = lgb.train(dict(BASE, seed=seed + k), lgb.Dataset(Xk, yk, weight=w[yk]), num_boost_round=n_trees)
        Pv += a[k] * bst.predict(S['X_val']); Pt += a[k] * bst.predict(S['X_test']); classes_seen.append(int(len(np.unique(yk))))
    out = score(S, Pv, Pt)
    return {'method': 'ens', 'seed': seed, 'n_trees': n_trees, 'history': [dict(round=1, **out)], 'final': out, 'classes_per_client': classes_seen,
            'client_sizes': n_k.astype(int).tolist(), 'minutes': (time.time() - t0) / 60, 'pred_test': Pt.argmax(1).astype(np.uint8)}

def run_cyc(S, parts, seed, cycles=30, iters_per_turn=1, eval_every=3):
    """Cyclic boosting: raw scores are carried as init_score, so each turn continues the shared model exactly."""
    t0 = time.time(); w = class_weights(S['y_train']); K = len(parts); rng = np.random.default_rng(seed)
    raw = [np.zeros((len(p), NC)) for p in parts]; Rv = np.zeros((len(S['y_val']), NC)); Rt = np.zeros((len(S['y_test']), NC))
    hist, n_iter = [], 0
    soft = lambda R: np.exp(R - R.max(1, keepdims=True)) / np.exp(R - R.max(1, keepdims=True)).sum(1, keepdims=True)
    for c in range(1, cycles + 1):
        for k in rng.permutation(K):
            p = parts[k]; Xk, yk = S['X_train'][p], S['y_train'][p]
            bst = lgb.train(dict(BASE, seed=seed * 1000 + n_iter), lgb.Dataset(Xk, yk, weight=w[yk], init_score=raw[k]),
                            num_boost_round=iters_per_turn)
            for j, q in enumerate(parts): raw[j] += bst.predict(S['X_train'][q], raw_score=True)
            Rv += bst.predict(S['X_val'], raw_score=True); Rt += bst.predict(S['X_test'], raw_score=True); n_iter += iters_per_turn
        if c == 1 or c % eval_every == 0 or c == cycles: hist.append(dict(round=c, iterations=n_iter, **score(S, soft(Rv), soft(Rt))))
    return {'method': 'cyc', 'seed': seed, 'cycles': cycles, 'iterations': n_iter, 'history': hist, 'final': hist[-1],
            'client_sizes': [len(p) for p in parts], 'minutes': (time.time() - t0) / 60, 'pred_test': Rt.argmax(1).astype(np.uint8)}
'''
open(f'{FT}/code/fedtree_core.py', 'w').write(TREE_CORE); open('/content/fedtree_core.py', 'w').write(TREE_CORE)
import fl_core_t as fc, fedtree_core as ftc; importlib.reload(fc); importlib.reload(ftc)
print('LightGBM', lgb.__version__, '| CPUs', os.cpu_count())

SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]; K = 10
SETTINGS = [('random', 'dirichlet', 0.1), ('random', 'ip', None), ('random', 'iid', None),
            ('random', 'dirichlet', 0.5), ('time', 'dirichlet', 0.1), ('time', 'ip', None)]   # the benchmark grid settings
src_ip = pd.read_parquet(f'{D}/raw_sampled.parquet', columns=['Src IP'])['Src IP'].astype(str).values
DATA = {}
def data(sp):
    if sp not in DATA:
        S = dict(np.load(f'{V2D}/split_{sp}.npz')); S['ip_train'] = src_ip[S['idx_train']]
        for k in ('train', 'val', 'test'): S[f'X_{k}'] = S[f'X_{k}'].astype(np.float32); S[f'y_{k}'] = S[f'y_{k}'].astype(int)
        DATA[sp] = S
    return DATA[sp]
parts_of = lambda sp, kind, a, s: fc.partition(data(sp)['y_train'], kind, K, s, alpha=a, groups=data(sp)['ip_train'] if kind == 'ip' else None)

# ---- Check 1: every client partition is identical to the benchmark grid's (partition signature of the FedAvg run) ----
bad = 0
for sp, kind, a in SETTINGS:
    tag = f'{sp}_{kind}{a if a else ""}'
    for s in SEEDS:
        ref = json.load(open(f'{RV2}/grid/{tag}_fedavg_s{s}.json'))['partition_sig']
        bad += fc.partition_sig(parts_of(sp, kind, a, s)) != ref
print(f'Check 1: client partitions identical to the benchmark grid in {len(SETTINGS) * len(SEEDS) - bad}/{len(SETTINGS) * len(SEEDS)} setting-seed pairs')
assert bad == 0, 'partitions differ - STOP and paste this output to Claude'

# ---- Check 2: centralized LightGBM (the lossless-histogram reference) reproduces the stored benchmark run ----
fp = f'{FT}/check/central_lgbm_random_s42.json'
if not os.path.exists(fp):
    S = data('random'); t0 = time.time()
    g = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1).fit(S['X_train'], S['y_train'])
    json.dump({'test_macro_f1': ftc.mf1(S['y_test'], g.predict(S['X_test'])), 'minutes': (time.time() - t0) / 60}, open(fp, 'w'))
c2 = json.load(open(fp)); st = json.load(open(f'{RV2}/central_lgbm_random_s42.json'))['test_macro_f1']
print(f'Check 2: centralized LightGBM test macro-F1 {c2["test_macro_f1"]:.4f} vs stored {st:.4f} (|diff| {abs(c2["test_macro_f1"] - st):.5f}) | {c2["minutes"]:.1f} min')

# ---- Timing run: both federated methods on random split, Dirichlet 0.5, seed 42 (kept as check files, not results) ----
for m, fn in (('ens', ftc.run_ens), ('cyc', ftc.run_cyc)):
    fp = f'{FT}/check/{m}_random_dirichlet0.5_s42.json'
    if not os.path.exists(fp):
        r = fn(data('random'), parts_of('random', 'dirichlet', 0.5, 42), 42); r.pop('pred_test'); json.dump(r, open(fp, 'w'))
    r = json.load(open(fp)); f = r['final']
    print(f'Timing {m}: val macro-F1 {f["val_macro_f1"]:.4f} | classes predicted {f["n_pred_classes"]} | {r["minutes"]:.1f} min per run')
print(f'harness saved to results/fedtrees/code/fedtree_core.py | grid estimate: {len(SETTINGS) * len(SEEDS)} runs per method | CELL 1 DONE')

In [ ]:
# FEDTREES — CELL 2: MAIN GRID (CPU, ~8.5 h for 120 runs; fully resumable - rerun the same cell after any disconnect).
#   6 benchmark settings x 10 seeds x {ens, cyc}; identical client partitions to the benchmark grid (checked in Cell 1).
#   Hyperparameters are the fixed centralized LightGBM settings (300 iterations, lr 0.05, 63 leaves, balanced weights);
#   nothing is tuned on test data. Output: results/fedtrees/grid/*.json (+ final test predictions as .npy).
#   The summary at the end sets the results beside the benchmark's federated FA-MLP (FedAvg, FedDyn; same partitions,
#   same seeds, mean of rounds 80/90/100) and the centralized LightGBM. Prints relative paths only.
import os, glob, json, time, sys, importlib
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
import lightgbm as lgb
from scipy import stats
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
RV2 = f'{DRIVE_PATH}/results/v2'; FT = f'{DRIVE_PATH}/results/fedtrees'; GR = f'{FT}/grid'; os.makedirs(GR, exist_ok=True)
assert os.path.exists(f'{FT}/check/cyc_random_dirichlet0.5_s42.json'), 'run Cell 1 first'
open('/content/fl_core_t.py', 'w').write(open(f'{V2D}/fl_core.py').read())
open('/content/fedtree_core.py', 'w').write(open(f'{FT}/code/fedtree_core.py').read()); sys.path.insert(0, '/content')
import fl_core_t as fc, fedtree_core as ftc; importlib.reload(fc); importlib.reload(ftc)

SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]; K = 10
SETTINGS = [('random', 'dirichlet', 0.5), ('random', 'dirichlet', 0.1), ('random', 'iid', None), ('random', 'ip', None),
            ('time', 'dirichlet', 0.1), ('time', 'ip', None)]
EST = {'ens': 6.1, 'cyc': 2.2}                                     # minutes per run, measured in Cell 1
src_ip = pd.read_parquet(f'{D}/raw_sampled.parquet', columns=['Src IP'])['Src IP'].astype(str).values
DATA = {}
def data(sp):
    if sp not in DATA:
        S = dict(np.load(f'{V2D}/split_{sp}.npz')); S['ip_train'] = src_ip[S['idx_train']]
        for k in ('train', 'val', 'test'): S[f'X_{k}'] = S[f'X_{k}'].astype(np.float32); S[f'y_{k}'] = S[f'y_{k}'].astype(int)
        DATA[sp] = S
    return DATA[sp]
tag_of = lambda sp, kind, a: f'{sp}_{kind}{a if a else ""}'
JOBS = [(sp, kind, a, s, m) for sp, kind, a in SETTINGS for s in SEEDS for m in ('cyc', 'ens')]
fp_of = lambda j: f'{GR}/{tag_of(*j[:3])}_{j[4]}_s{j[3]}.json'
todo = [j for j in JOBS if not os.path.exists(fp_of(j))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{sum(EST[j[4]] for j in todo) / 60:.1f} h left')
t0 = time.time()
for i, j in enumerate(todo):
    sp, kind, a, s, m = j; S = data(sp)
    parts = fc.partition(S['y_train'], kind, K, s, alpha=a, groups=S['ip_train'] if kind == 'ip' else None)
    ref_sig = json.load(open(f'{RV2}/grid/{tag_of(sp, kind, a)}_fedavg_s{s}.json'))['partition_sig']
    assert fc.partition_sig(parts) == ref_sig, 'partition differs from the benchmark grid - STOP'
    r = (ftc.run_cyc if m == 'cyc' else ftc.run_ens)(S, parts, s)
    np.save(fp_of(j).replace('.json', '_pred.npy'), r.pop('pred_test')); r['partition_sig'] = ref_sig; r['setting'] = tag_of(sp, kind, a)
    json.dump(r, open(fp_of(j), 'w'))
    left = sum(EST[x[4]] for x in todo[i + 1:]) / 60
    print(f'  [{i + 1}/{len(todo)}] {tag_of(sp, kind, a):<20} {m} s{s:<5} test macro-F1 {r["final"]["test_macro_f1"]:.4f} | '
          f'{r["minutes"]:.1f} min | ~{left:.1f} h left', flush=True)

# ---- summary over finished runs (works on a partial grid) ----
L3 = lambda o: float(np.mean([h['test_macro_f1'] for h in o['history'] if h['round'] in (80, 90, 100)]))
rows = []
for sp, kind, a in SETTINGS:
    tg = tag_of(sp, kind, a)
    for s in SEEDS:
        row = {'setting': tg, 'seed': s}
        for m in ('ens', 'cyc'):
            f = f'{GR}/{tg}_{m}_s{s}.json'
            row[m] = json.load(open(f))['final']['test_macro_f1'] if os.path.exists(f) else np.nan
        for m in ('fedavg', 'feddyn'):
            f = f'{RV2}/grid/{tg}_{m}_s{s}.json'
            row[m] = L3(json.load(open(f))) if os.path.exists(f) else np.nan
        rows.append(row)
df = pd.DataFrame(rows); df.to_csv(f'{FT}/grid_summary.csv', index=False)
pd.set_option('display.width', 200)
print('\n===== test macro-F1, mean (SD) over seeds; FA-MLP = benchmark federated neural runs on the same partitions =====')
for tg, g in df.groupby('setting', sort=False):
    cells = ' | '.join(f'{m} {g[m].mean():.4f} ({g[m].std():.4f}, n={g[m].notna().sum()})' for m in ('ens', 'cyc', 'fedavg', 'feddyn'))
    print(f'  {tg:<20} {cells}')
print('\n===== paired differences over seeds (pp), tree method minus FA-MLP FedAvg =====')
for tg, g in df.groupby('setting', sort=False):
    for m in ('ens', 'cyc'):
        p = g[[m, 'fedavg']].dropna()
        if len(p) >= 2:
            d = 100 * (p[m] - p.fedavg); se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
            print(f'  {tg:<20} {m} - fedavg: {d.mean():+.2f} pp [{d.mean() - t * se:+.2f}, {d.mean() + t * se:+.2f}] '
                  f'(n={len(d)}, won {int((d > 0).sum())}, p {stats.ttest_rel(p[m], p.fedavg).pvalue:.4f})')
for sp in ('random', 'time'):
    f = f'{RV2}/central_lgbm_{sp}_s42.json'
    if os.path.exists(f): print(f'centralized LightGBM ({sp} split, lossless-histogram reference): {json.load(open(f))["test_macro_f1"]:.4f}')
done = sum(os.path.exists(fp_of(j)) for j in JOBS)
print(f'\nruns finished {done}/{len(JOBS)} | saved results/fedtrees/grid_summary.csv | session {(time.time() - t0) / 60:.1f} min'
      + (' | CELL 2 DONE' if done == len(JOBS) else ' | grid not complete - rerun this cell'))

In [ ]:
# FEDTREES — CELL 3: DIAGNOSIS + EXPORT (CPU, ~3-5 min). Reads results only; trains nothing.
#   1) per-client class counts for every setting and seed (recomputed from the same partitions, signature-checked)
#   2) per-class test F1 of ens / cyc / FA-MLP FedAvg, mean over seeds - shows WHICH classes the ensemble loses
#   3) zips results/fedtrees (code, checks, grid JSONs + test predictions, summaries) + the test labels -> download
#   Prints relative paths only.
import os, glob, json, sys, importlib, zipfile
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
RV2 = f'{DRIVE_PATH}/results/v2'; FT = f'{DRIVE_PATH}/results/fedtrees'; GR = f'{FT}/grid'
open('/content/fl_core_t.py', 'w').write(open(f'{V2D}/fl_core.py').read()); sys.path.insert(0, '/content')
import fl_core_t as fc; importlib.reload(fc)
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]; K = 10; NC = 8
SETTINGS = [('random', 'dirichlet', 0.5), ('random', 'dirichlet', 0.1), ('random', 'iid', None), ('random', 'ip', None),
            ('time', 'dirichlet', 0.1), ('time', 'ip', None)]
CLS = ['Benign', 'Brute Force', 'DDOS', 'DOS', 'Mirai', 'Recon', 'Spoofing', 'Web-Based']
tag_of = lambda sp, kind, a: f'{sp}_{kind}{a if a else ""}'
src_ip = pd.read_parquet(f'{D}/raw_sampled.parquet', columns=['Src IP'])['Src IP'].astype(str).values
os.makedirs(f'{FT}/diag', exist_ok=True); rows = []
for sp in ('random', 'time'):
    S = np.load(f'{V2D}/split_{sp}.npz'); y = S['y_train'].astype(int); ip = src_ip[S['idx_train']]
    np.save(f'{FT}/diag/y_test_{sp}.npy', S['y_test'].astype(np.uint8))
    for _, kind, a in [s for s in SETTINGS if s[0] == sp]:
        tg = tag_of(sp, kind, a)
        for s in SEEDS:
            parts = fc.partition(y, kind, K, s, alpha=a, groups=ip if kind == 'ip' else None)
            assert fc.partition_sig(parts) == json.load(open(f'{GR}/{tg}_ens_s{s}.json'))['partition_sig'], 'partition differs - STOP'
            for k, p in enumerate(parts):
                rows.append(dict(setting=tg, seed=s, client=k, n=len(p), **{c: int(v) for c, v in zip(CLS, np.bincount(y[p], minlength=NC))}))
cc = pd.DataFrame(rows); cc.to_csv(f'{FT}/diag/client_class_counts.csv', index=False)
cc['n_classes'] = (cc[CLS] > 0).sum(1); cc['top_share'] = cc[CLS].max(1) / cc.n
print('===== clients: classes held (min / median / max) and share of the largest class (median) =====')
for tg, g in cc.groupby('setting', sort=False):
    print(f'  {tg:<20} classes {g.n_classes.min()} / {g.n_classes.median():.0f} / {g.n_classes.max()} | largest-class share {g.top_share.median():.2f} '
          f'| clients holding each class (median over seeds): ' + ' '.join(f'{c[:4]} {int(g.groupby("seed")[c].apply(lambda v: (v > 0).sum()).median())}' for c in CLS))
L3pc = lambda o, c: float(np.mean([h['test_per_class_f1'][c] for h in o['history'] if h['round'] in (80, 90, 100)]))
print('\n===== per-class test F1, mean over seeds: ens | cyc | FA-MLP FedAvg =====')
print(f'  {"setting":<20} ' + ' '.join(f'{c[:11]:>17}' for c in CLS))
for sp, kind, a in SETTINGS:
    tg = tag_of(sp, kind, a)
    E = np.mean([json.load(open(f'{GR}/{tg}_ens_s{s}.json'))['final']['test_per_class_f1'] for s in SEEDS], 0)
    Cy = np.mean([json.load(open(f'{GR}/{tg}_cyc_s{s}.json'))['final']['test_per_class_f1'] for s in SEEDS], 0)
    F = np.mean([[L3pc(json.load(open(f'{RV2}/grid/{tg}_fedavg_s{s}.json')), c) for c in range(NC)] for s in SEEDS], 0)
    print(f'  {tg:<20} ' + ' '.join(f'{e:.2f}|{c:.2f}|{f:.2f}'.rjust(17) for e, c, f in zip(E, Cy, F)))
zp = f'{FT}/export_fedtrees.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{FT}/**/*', recursive=True)):
        if os.path.isfile(f) and not f.endswith('.zip'): z.write(f, os.path.relpath(f, FT)); n += 1
print(f'\nexported {n} files ({os.path.getsize(zp) / 1e6:.1f} MB) -> results/fedtrees/export_fedtrees.zip | CELL 3 DONE'); files.download(zp)

In [ ]:
# FEDTREES — CELL 4: HISTOGRAM-AGGREGATION FEDERATED GBDT ("hist"), measured instead of argued (CPU, ~2.5-3 h, resumable).
#   Protocol (horizontal federated GBDT, as in FedTree / federated XGBoost):
#     1) federated binning: each client sends 256 local quantiles per feature; the server pools them (weight n_k / 257
#        per point) and takes 255 weighted quantiles -> one global set of bin edges per feature (<= 255 bins)
#     2) training: for every node, clients send gradient/hessian histograms over these bins; the server sums them and
#        picks the split. Summed histograms equal the histograms of the pooled, binned data, so training LightGBM on the
#        pooled data mapped to these bin indices (one LightGBM bin per index, min_data_in_bin=1) is exactly this protocol.
#   Same LightGBM settings as Cells 1-2 (300 iterations, lr 0.05, 63 leaves, balanced global class weights); nothing tuned.
#   Also: 2 binning checks (pooled-data edges, no federation) to separate the binning effect, and per-tree leaf counts
#   for the communication-cost analysis. Output: results/fedtrees/grid_hist/*.json (+ test predictions). Relative paths only.
import os, glob, json, time, sys, importlib
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
import lightgbm as lgb
from scipy import stats
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
RV2 = f'{DRIVE_PATH}/results/v2'; FT = f'{DRIVE_PATH}/results/fedtrees'; GR = f'{FT}/grid'; GH = f'{FT}/grid_hist'
os.makedirs(GH, exist_ok=True)
open('/content/fl_core_t.py', 'w').write(open(f'{V2D}/fl_core.py').read())
open('/content/fedtree_core.py', 'w').write(open(f'{FT}/code/fedtree_core.py').read()); sys.path.insert(0, '/content')
import fl_core_t as fc, fedtree_core as ftc; importlib.reload(fc); importlib.reload(ftc)
NC = ftc.NC; Q = 256; MAXB = 255

def client_sketch(X):                                     # what one client sends: Q+1 quantiles per feature
    return np.quantile(X, np.linspace(0, 1, Q + 1), axis=0)          # (Q+1) x F

def merge_edges(sketches, sizes):
    """Server: pool the client quantile points (weight n_k/(Q+1) each) and take MAXB weighted quantiles per feature."""
    P = np.concatenate(sketches, 0); W = np.concatenate([np.full(len(s), n / len(s)) for s, n in zip(sketches, sizes)])
    edges = []
    for f in range(P.shape[1]):
        o = np.argsort(P[:, f], kind='stable'); v, cw = P[o, f], np.cumsum(W[o]); cw /= cw[-1]
        cut = v[np.minimum(np.searchsorted(cw, np.linspace(0, 1, MAXB + 1)[1:-1]), len(v) - 1)]
        edges.append(np.unique(cut))
    return edges

def to_bins(X, edges):                                    # bin i = (e_{i-1}, e_i]; done locally by each client
    return np.stack([np.searchsorted(e, X[:, f], side='left') for f, e in enumerate(edges)], 1).astype(np.float32)

def run_hist(S, edges, seed, n_trees=300):
    t0 = time.time(); w = ftc.class_weights(S['y_train']); Xb = to_bins(S['X_train'], edges)
    prm = dict(ftc.BASE, seed=seed, max_bin=MAXB, min_data_in_bin=1, bin_construct_sample_cnt=len(Xb), feature_pre_filter=False)
    ds = lgb.Dataset(Xb, S['y_train'], weight=w[S['y_train']], params=prm).construct()
    nd = [len(np.unique(Xb[:, f])) for f in range(Xb.shape[1])]                  # exactness: one LightGBM bin per index
    assert all(ds.feature_num_bin(f) == (v if v > 1 else 0) for f, v in enumerate(nd)), 'LightGBM merged bins - STOP'
    bst = lgb.train(prm, ds, num_boost_round=n_trees)
    Pv, Pt = bst.predict(to_bins(S['X_val'], edges)), bst.predict(to_bins(S['X_test'], edges)); out = ftc.score(S, Pv, Pt)
    leaves = [t['num_leaves'] for t in bst.dump_model()['tree_info']]
    return {'method': 'hist', 'seed': seed, 'n_trees': n_trees, 'final': out, 'history': [dict(round=n_trees, **out)],
            'bins_per_feature': [int(len(e) + 1) for e in edges], 'leaves_per_tree': leaves, 'n_features': int(Xb.shape[1]),
            'minutes': (time.time() - t0) / 60, 'pred_test': Pt.argmax(1).astype(np.uint8)}

SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]; K = 10
SETTINGS = [('random', 'ip', None), ('time', 'ip', None), ('random', 'dirichlet', 0.1), ('time', 'dirichlet', 0.1),
            ('random', 'iid', None), ('random', 'dirichlet', 0.5)]
tag_of = lambda sp, kind, a: f'{sp}_{kind}{a if a else ""}'
src_ip = pd.read_parquet(f'{D}/raw_sampled.parquet', columns=['Src IP'])['Src IP'].astype(str).values
DATA = {}
def data(sp):
    if sp not in DATA:
        S = dict(np.load(f'{V2D}/split_{sp}.npz')); S['ip_train'] = src_ip[S['idx_train']]
        for k in ('train', 'val', 'test'): S[f'X_{k}'] = S[f'X_{k}'].astype(np.float32); S[f'y_{k}'] = S[f'y_{k}'].astype(int)
        DATA[sp] = S
    return DATA[sp]

# ---- binning checks: pooled-data quantile edges (one "client" = all data), one per split ----
for sp in ('random', 'time'):
    f = f'{GH}/check_pooled_edges_{sp}.json'
    if not os.path.exists(f):
        S = data(sp); r = run_hist(S, merge_edges([client_sketch(S['X_train'])], [len(S['y_train'])]), 42); r.pop('pred_test'); json.dump(r, open(f, 'w'))
    r = json.load(open(f)); c = json.load(open(f'{RV2}/central_lgbm_{sp}_s42.json'))['test_macro_f1']
    print(f'binning check {sp}: pooled-edge hist {r["final"]["test_macro_f1"]:.4f} vs centralized LightGBM {c:.4f} | {r["minutes"]:.1f} min')

JOBS = [(sp, kind, a, s) for sp, kind, a in SETTINGS for s in SEEDS]
fp_of = lambda j: f'{GH}/{tag_of(*j[:3])}_hist_s{j[3]}.json'
todo = [j for j in JOBS if not os.path.exists(fp_of(j))]
est = float(np.mean([json.load(open(f))['minutes'] for f in glob.glob(f'{GH}/check_pooled_edges_*.json')]))
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{len(todo) * est / 60:.1f} h left')
t0 = time.time()
for i, j in enumerate(todo):
    sp, kind, a, s = j; S = data(sp); tg = tag_of(sp, kind, a)
    parts = fc.partition(S['y_train'], kind, K, s, alpha=a, groups=S['ip_train'] if kind == 'ip' else None)
    sig = json.load(open(f'{GR}/{tg}_ens_s{s}.json'))['partition_sig']
    assert fc.partition_sig(parts) == sig, 'partition differs from Cell 2 - STOP'
    edges = merge_edges([client_sketch(S['X_train'][p]) for p in parts], [len(p) for p in parts])
    r = run_hist(S, edges, s); np.save(fp_of(j).replace('.json', '_pred.npy'), r.pop('pred_test'))
    r['partition_sig'] = sig; r['setting'] = tg; json.dump(r, open(fp_of(j), 'w'))
    print(f'  [{i + 1}/{len(todo)}] {tg:<20} s{s:<5} test macro-F1 {r["final"]["test_macro_f1"]:.4f} | {r["minutes"]:.1f} min | '
          f'~{(len(todo) - i - 1) * est / 60:.1f} h left', flush=True)

# ---- summary over finished runs (works on a partial grid) ----
L3 = lambda o: float(np.mean([h['test_macro_f1'] for h in o['history'] if h['round'] in (80, 90, 100)]))
rows = []
for sp, kind, a in SETTINGS:
    tg = tag_of(sp, kind, a)
    for s in SEEDS:
        f = f'{GH}/{tg}_hist_s{s}.json'
        if not os.path.exists(f): continue
        row = {'setting': tg, 'seed': s, 'hist': json.load(open(f))['final']['test_macro_f1']}
        for m in ('ens', 'cyc'): row[m] = json.load(open(f'{GR}/{tg}_{m}_s{s}.json'))['final']['test_macro_f1']
        for m in ('fedavg', 'feddyn'): row[m] = L3(json.load(open(f'{RV2}/grid/{tg}_{m}_s{s}.json')))
        rows.append(row)
df = pd.DataFrame(rows); df.to_csv(f'{FT}/grid_hist_summary.csv', index=False)
if len(df):
    print('\n===== test macro-F1, mean (SD) over seeds; centralized LightGBM: random 0.6681, time 0.4318 =====')
    for tg, g in df.groupby('setting', sort=False):
        print(f'  {tg:<20} n={len(g):<2} ' + ' | '.join(f'{m} {g[m].mean():.4f} ({g[m].std():.4f})' for m in ('hist', 'ens', 'cyc', 'fedavg', 'feddyn')))
    print('\n===== paired differences over seeds (pp), hist minus the other method =====')
    for tg, g in df.groupby('setting', sort=False):
        if len(g) < 2: continue
        for m in ('fedavg', 'feddyn', 'cyc', 'ens'):
            d = 100 * (g['hist'] - g[m]); se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
            print(f'  {tg:<20} hist - {m:<6}: {d.mean():+.2f} pp [{d.mean() - t * se:+.2f}, {d.mean() + t * se:+.2f}] '
                  f'(n={len(d)}, won {int((d > 0).sum())}, p {stats.ttest_rel(g["hist"], g[m]).pvalue:.4f})')
done = sum(os.path.exists(fp_of(j)) for j in JOBS)
print(f'\nruns finished {done}/{len(JOBS)} | saved results/fedtrees/grid_hist_summary.csv | session {(time.time() - t0) / 60:.1f} min'
      + (' | CELL 4 DONE' if done == len(JOBS) else ' | grid not complete - rerun this cell'))

In [ ]:
# FEDTREES — CELL 5: FINAL EXPORT (CPU, ~1 min). Trains nothing.
#   Zips everything under results/fedtrees (code, checks, diag, grid, grid_hist, summaries) -> download. Relative paths only.
import os, glob, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; FT = f'{cands[0]}/results/fedtrees'
n_g, n_h = len(glob.glob(f'{FT}/grid/*_s*.json')), len(glob.glob(f'{FT}/grid_hist/*_hist_s*.json'))
assert n_g == 120 and n_h == 60, f'grid {n_g}/120, grid_hist {n_h}/60 - finish Cells 2 and 4 first'
zp = f'{FT}/export_fedtrees_final.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{FT}/**/*', recursive=True)):
        if os.path.isfile(f) and not f.endswith('.zip'): z.write(f, os.path.relpath(f, FT)); n += 1
subdirs = sorted({os.path.relpath(f, FT).split('/')[0] for f in glob.glob(f'{FT}/**/*', recursive=True) if os.path.isfile(f) and not f.endswith('.zip')})
print(f'exported {n} files ({os.path.getsize(zp) / 1e6:.1f} MB) from {subdirs} -> results/fedtrees/export_fedtrees_final.zip | CELL 5 DONE'); files.download(zp)

In [ ]:
# FEDTREES — CELL 6: REVIEWER CHECKS (CPU, ~6 h for 90 runs; fully resumable, priority order).
#   A) Are the model-sharing failures under source-IP clients an artifact of their settings? (random_ip, time_ip, 10 seeds)
#        cyc5  : cyclic boosting with 5 boosting iterations per client turn (6 cycles = 300 iterations, 5x fewer turns)
#        cyc10 : 10 iterations per turn (3 cycles = 300 iterations, 10x fewer turns)
#        ens100: one-shot ensemble with 100 instead of 300 iterations per client
#   B) Scaling to 50 clients (random split, Dirichlet 0.5, the partitions of the benchmark's 50-client runs, 10 seeds):
#        ens, cyc (6 cycles x 50 clients = 300 iterations), hist -- beside the benchmark's 50-client FA-MLP FedAvg/FedDyn
#   Step 0 re-runs one Cell 4 hist run from the saved hist module to prove the module is the code that produced Cell 4.
#   Output: results/fedtrees/grid_extra/*.json (+ test predictions), results/fedtrees/code/fedtree_hist.py. Relative paths only.
import os, glob, json, time, sys, importlib
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
import lightgbm as lgb
from scipy import stats
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
RV2 = f'{DRIVE_PATH}/results/v2'; PP = f'{RV2}/grid_partial'; FT = f'{DRIVE_PATH}/results/fedtrees'
GR, GH, GX = f'{FT}/grid', f'{FT}/grid_hist', f'{FT}/grid_extra'; os.makedirs(GX, exist_ok=True)

HIST_CORE = r'''
"""fedtree_hist.py - histogram-aggregation federated GBDT (federated quantile binning + summed gradient histograms).
Training LightGBM on the pooled data mapped to the global bin indices (one LightGBM bin per index) is exactly the protocol."""
import time, numpy as np, lightgbm as lgb
import fedtree_core as ftc
Q = 256; MAXB = 255

def client_sketch(X):                                     # what one client sends: Q+1 quantiles per feature
    return np.quantile(X, np.linspace(0, 1, Q + 1), axis=0)

def merge_edges(sketches, sizes):                         # server: weighted quantiles of the pooled client sketches
    P = np.concatenate(sketches, 0); W = np.concatenate([np.full(len(s), n / len(s)) for s, n in zip(sketches, sizes)])
    edges = []
    for f in range(P.shape[1]):
        o = np.argsort(P[:, f], kind='stable'); v, cw = P[o, f], np.cumsum(W[o]); cw /= cw[-1]
        cut = v[np.minimum(np.searchsorted(cw, np.linspace(0, 1, MAXB + 1)[1:-1]), len(v) - 1)]
        edges.append(np.unique(cut))
    return edges

def to_bins(X, edges):                                    # bin i = (e_{i-1}, e_i]; done locally by each client
    return np.stack([np.searchsorted(e, X[:, f], side='left') for f, e in enumerate(edges)], 1).astype(np.float32)

def run_hist(S, edges, seed, n_trees=300):
    t0 = time.time(); w = ftc.class_weights(S['y_train']); Xb = to_bins(S['X_train'], edges)
    prm = dict(ftc.BASE, seed=seed, max_bin=MAXB, min_data_in_bin=1, bin_construct_sample_cnt=len(Xb), feature_pre_filter=False)
    ds = lgb.Dataset(Xb, S['y_train'], weight=w[S['y_train']], params=prm).construct()
    nd = [len(np.unique(Xb[:, f])) for f in range(Xb.shape[1])]
    assert all(ds.feature_num_bin(f) == (v if v > 1 else 0) for f, v in enumerate(nd)), 'LightGBM merged bins - STOP'
    bst = lgb.train(prm, ds, num_boost_round=n_trees)
    Pv, Pt = bst.predict(to_bins(S['X_val'], edges)), bst.predict(to_bins(S['X_test'], edges)); out = ftc.score(S, Pv, Pt)
    leaves = [t['num_leaves'] for t in bst.dump_model()['tree_info']]
    return {'method': 'hist', 'seed': seed, 'n_trees': n_trees, 'final': out, 'history': [dict(round=n_trees, **out)],
            'bins_per_feature': [int(len(e) + 1) for e in edges], 'leaves_per_tree': leaves, 'n_features': int(Xb.shape[1]),
            'minutes': (time.time() - t0) / 60, 'pred_test': Pt.argmax(1).astype(np.uint8)}
'''
open(f'{FT}/code/fedtree_hist.py', 'w').write(HIST_CORE); open('/content/fedtree_hist.py', 'w').write(HIST_CORE)
open('/content/fl_core_t.py', 'w').write(open(f'{V2D}/fl_core.py').read())
open('/content/fedtree_core.py', 'w').write(open(f'{FT}/code/fedtree_core.py').read()); sys.path.insert(0, '/content')
import fl_core_t as fc, fedtree_core as ftc, fedtree_hist as fth
for m in (fc, ftc, fth): importlib.reload(m)

SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
tag_of = lambda sp, kind, a: f'{sp}_{kind}{a if a else ""}'
src_ip = pd.read_parquet(f'{D}/raw_sampled.parquet', columns=['Src IP'])['Src IP'].astype(str).values
DATA = {}
def data(sp):
    if sp not in DATA:
        S = dict(np.load(f'{V2D}/split_{sp}.npz')); S['ip_train'] = src_ip[S['idx_train']]
        for k in ('train', 'val', 'test'): S[f'X_{k}'] = S[f'X_{k}'].astype(np.float32); S[f'y_{k}'] = S[f'y_{k}'].astype(int)
        DATA[sp] = S
    return DATA[sp]

# ---- Step 0: the saved hist module reproduces the Cell 4 run (random split, Dirichlet 0.5, seed 42) ----
f0 = f'{GX}/check_hist_module_random_dirichlet0.5_s42.json'
if not os.path.exists(f0):
    S = data('random'); parts = fc.partition(S['y_train'], 'dirichlet', 10, 42, alpha=0.5)
    r = fth.run_hist(S, fth.merge_edges([fth.client_sketch(S['X_train'][p]) for p in parts], [len(p) for p in parts]), 42)
    r.pop('pred_test'); json.dump(r, open(f0, 'w'))
r0, c4 = json.load(open(f0)), json.load(open(f'{GH}/random_dirichlet0.5_hist_s42.json'))
d0 = abs(r0['final']['test_macro_f1'] - c4['final']['test_macro_f1'])
print(f'Step 0: hist module {r0["final"]["test_macro_f1"]:.6f} vs Cell 4 {c4["final"]["test_macro_f1"]:.6f} (|diff| {d0:.6f}) | {r0["minutes"]:.1f} min')
assert d0 < 1e-4, 'hist module does not reproduce Cell 4 - STOP and paste this output'

# ---- jobs (priority order) ----
JOBS = [('A', sp, kind, v, s) for sp, kind in (('random', 'ip'), ('time', 'ip')) for v in ('cyc5', 'cyc10', 'ens100') for s in SEEDS]
JOBS += [('B', 'random', 'dirichlet0.5_K50', v, s) for v in ('hist', 'cyc', 'ens') for s in SEEDS]
EST = {'cyc5': 3.6, 'cyc10': 3.6, 'ens100': 3.6, 'hist': 2.4, 'cyc': 3.0, 'ens': 10.0}      # minutes, estimated
fp_of = lambda j: f'{GX}/{j[1]}_{j[2]}_{j[3]}_s{j[4]}.json'
def parts_of(j):
    _, sp, kind, v, s = j; S = data(sp)
    if j[0] == 'A':
        p = fc.partition(S['y_train'], 'ip', 10, s, groups=S['ip_train'])
        assert fc.partition_sig(p) == json.load(open(f'{GR}/{sp}_ip_ens_s{s}.json'))['partition_sig'], 'partition differs - STOP'
    else:
        p = fc.partition(S['y_train'], 'dirichlet', 50, s, alpha=0.5)
        ref = json.load(open(f'{PP}/random_dirichlet0.5_K50_m10_fedavg_s{s}.json'))['client_sizes']
        assert [len(q) for q in p] == ref, '50-client partition differs from the benchmark - STOP'
    return S, p
todo = [j for j in JOBS if not os.path.exists(fp_of(j))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{sum(EST[j[3]] for j in todo) / 60:.1f} h left')
t0 = time.time()
for i, j in enumerate(todo):
    S, p = parts_of(j); v, s = j[3], j[4]
    if v == 'cyc5': r = ftc.run_cyc(S, p, s, cycles=6, iters_per_turn=5, eval_every=3)
    elif v == 'cyc10': r = ftc.run_cyc(S, p, s, cycles=3, iters_per_turn=10, eval_every=3)
    elif v == 'cyc': r = ftc.run_cyc(S, p, s, cycles=6, iters_per_turn=1, eval_every=3)
    elif v == 'ens100': r = ftc.run_ens(S, p, s, n_trees=100)
    elif v == 'ens': r = ftc.run_ens(S, p, s)
    else: r = fth.run_hist(S, fth.merge_edges([fth.client_sketch(S['X_train'][q]) for q in p], [len(q) for q in p]), s)
    np.save(fp_of(j).replace('.json', '_pred.npy'), r.pop('pred_test')); r['partition_sig'] = fc.partition_sig(p)
    r['variant'] = v; r['setting'] = f'{j[1]}_{j[2]}'; r['K'] = len(p); json.dump(r, open(fp_of(j), 'w'))
    print(f'  [{i + 1}/{len(todo)}] {j[1]}_{j[2]:<18} {v:<7} s{s:<5} test macro-F1 {r["final"]["test_macro_f1"]:.4f} | {r["minutes"]:.1f} min | '
          f'~{sum(EST[x[3]] for x in todo[i + 1:]) / 60:.1f} h left', flush=True)

# ---- summary over finished runs (works on a partial grid) ----
def pr(a, b, lab):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2: return f'{lab}: n<2'
    d = 100 * (b[m] - a[m]); se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
    return (f'{lab}: {a[m].mean():.4f} -> {b[m].mean():.4f} = {d.mean():+.2f} pp [{d.mean() - t * se:+.2f}, {d.mean() + t * se:+.2f}] '
            f'(n={len(d)}, higher in {int((d > 0).sum())}, p {stats.ttest_rel(b[m], a[m]).pvalue:.4f})')
ld = lambda f, k='final': json.load(open(f))[k]['test_macro_f1'] if os.path.exists(f) else np.nan
L = lambda f, rounds: float(np.mean([h['test_macro_f1'] for h in json.load(open(f))['history'] if h['round'] in rounds])) if os.path.exists(f) else np.nan
print('\n===== A) model-sharing methods under source-IP clients: settings check (test macro-F1, paired over seeds) =====')
for sp in ('random', 'time'):
    base = {m: np.array([ld(f'{GR}/{sp}_ip_{m}_s{s}.json') for s in SEEDS]) for m in ('cyc', 'ens')}
    fa = np.array([L(f'{RV2}/grid/{sp}_ip_fedavg_s{s}.json', (80, 90, 100)) for s in SEEDS])
    hi = np.array([ld(f'{GH}/{sp}_ip_hist_s{s}.json') for s in SEEDS])
    for v, ref in (('cyc5', 'cyc'), ('cyc10', 'cyc'), ('ens100', 'ens')):
        x = np.array([ld(f'{GX}/{sp}_ip_{v}_s{s}.json') for s in SEEDS])
        print(f'  {sp}_ip {v:<7} ' + pr(base[ref], x, f'vs {ref} (Cell 2)') + '\n' + ' ' * 18 + pr(fa, x, 'vs FA-MLP FedAvg')
              + '\n' + ' ' * 18 + pr(x, hi, 'hist minus it'))
print('\n===== B) 50 clients (random split, Dirichlet 0.5): test macro-F1, mean over seeds =====')
for v in ('hist', 'cyc', 'ens'):
    x50 = np.array([ld(f'{GX}/random_dirichlet0.5_K50_{v}_s{s}.json') for s in SEEDS])
    x10 = np.array([ld(f'{GH if v == "hist" else GR}/random_dirichlet0.5_{v}_s{s}.json') for s in SEEDS])
    print(f'  {v:<5} ' + pr(x10, x50, '10 -> 50 clients'))
for m in ('fedavg', 'feddyn'):
    a = np.array([L(f'{PP}/random_dirichlet0.5_K50_m10_{m}_s{s}.json', (80, 90, 100)) for s in SEEDS])
    b = np.array([L(f'{PP}/random_dirichlet0.5_K50_m10_R500_{m}_s{s}.json', (480, 490, 500)) for s in SEEDS])
    print(f'  FA-MLP {m:<6} 50 clients, 10 per round: 100 rounds {np.nanmean(a):.4f} | 500 rounds {np.nanmean(b):.4f}')
done = sum(os.path.exists(fp_of(j)) for j in JOBS)
print(f'\nruns finished {done}/{len(JOBS)} | session {(time.time() - t0) / 60:.1f} min'
      + (' | CELL 6 DONE' if done == len(JOBS) else ' | grid not complete - rerun this cell'))

In [ ]:
# FEDTREES — CELL 7: FINAL EXPORT, re-run of Cell 5 after Cell 6 (CPU, ~1-2 min). Trains nothing.
#   Zips everything under results/fedtrees (code, checks, diag, grid, grid_hist, grid_extra, summaries) -> download.
#   Prints relative paths only.
import os, glob, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; FT = f'{cands[0]}/results/fedtrees'
n_g = len(glob.glob(f'{FT}/grid/*_s*.json'))
n_h = len(glob.glob(f'{FT}/grid_hist/*_hist_s*.json'))
n_x = len([f for f in glob.glob(f'{FT}/grid_extra/*_s*.json') if not os.path.basename(f).startswith('check_')])
assert n_g == 120 and n_h == 60 and n_x == 90, f'grid {n_g}/120, grid_hist {n_h}/60, grid_extra {n_x}/90 - finish Cells 2, 4 and 6 first'
assert os.path.exists(f'{FT}/code/fedtree_hist.py') and os.path.exists(f'{FT}/grid_extra/check_hist_module_random_dirichlet0.5_s42.json')
zp = f'{FT}/export_fedtrees_final.zip'
if os.path.exists(zp): os.remove(zp)
n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{FT}/**/*', recursive=True)):
        if os.path.isfile(f) and not f.endswith('.zip'): z.write(f, os.path.relpath(f, FT)); n += 1
subdirs = sorted({os.path.relpath(f, FT).split('/')[0] for f in glob.glob(f'{FT}/**/*', recursive=True) if os.path.isfile(f) and not f.endswith('.zip')})
print(f'grid {n_g} | grid_hist {n_h} | grid_extra {n_x}')
print(f'exported {n} files ({os.path.getsize(zp) / 1e6:.1f} MB) from {subdirs} -> results/fedtrees/export_fedtrees_final.zip | CELL 5 DONE'); files.download(zp)